# 03 - RDS Gold Layer Queries

Query the RDS PostgreSQL database (Azure PostgreSQL) that holds the gold-layer
output: `custom_recommendations`, `finops_threshold_rules`, `service_configuration`,
etc.

Use this to:
- Validate gold-layer data written by the pipeline
- Debug recommendation deduplication
- Analyze savings by cloud/service/account
- Compare silver (Iceberg) vs gold (RDS) data

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from local_helpers import rds_query, rds_execute, get_rds_engine
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

## 1. Schema Overview

In [ ]:
# List all tables
rds_query("""
    SELECT table_name, 
           pg_size_pretty(pg_total_relation_size(quote_ident(table_name))) as size
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    ORDER BY table_name
""")

In [ ]:
# Describe custom_recommendations table
rds_query("""
    SELECT column_name, data_type, is_nullable
    FROM information_schema.columns
    WHERE table_name = 'custom_recommendations'
    ORDER BY ordinal_position
""")

## 2. Recommendations Overview

In [ ]:
# Count by cloud provider and rec_type
rds_query("""
    SELECT cloud_provider, rec_type, COUNT(*) as cnt,
           ROUND(SUM(CAST(savings AS numeric)), 2) as total_savings
    FROM custom_recommendations
    GROUP BY cloud_provider, rec_type
    ORDER BY total_savings DESC
""")

In [ ]:
# Recent recommendations
rds_query("""
    SELECT resource_id, cloud_provider, rec_type, recommendation,
           category, savings, region, recommendation_date
    FROM custom_recommendations
    ORDER BY recommendation_date DESC
    LIMIT 20
""")

In [ ]:
# Recommendations with JSON details
rds_query("""
    SELECT resource_id, recommendation, savings, details::text
    FROM custom_recommendations
    WHERE details IS NOT NULL AND details != ''
    ORDER BY recommendation_date DESC
    LIMIT 10
""")

## 3. Savings Analysis

In [ ]:
# Total savings by cloud
rds_query("""
    SELECT cloud_provider,
           COUNT(*) as recommendations,
           ROUND(SUM(CAST(savings AS numeric)), 2) as total_monthly_savings,
           ROUND(SUM(CAST(savings AS numeric)) * 12, 2) as total_annual_savings,
           ROUND(AVG(CAST(savings AS numeric)), 2) as avg_savings_per_rec
    FROM custom_recommendations
    WHERE savings IS NOT NULL
    GROUP BY cloud_provider
    ORDER BY total_monthly_savings DESC
""")

In [ ]:
# Top 20 highest-savings recommendations
rds_query("""
    SELECT resource_id, cloud_provider, category, rec_type,
           recommendation, savings, region
    FROM custom_recommendations
    WHERE savings IS NOT NULL
    ORDER BY CAST(savings AS numeric) DESC
    LIMIT 20
""")

## 4. Threshold Rules

In [ ]:
# View threshold rules (if table exists)
try:
    display(rds_query("""
        SELECT * FROM finops_threshold_rules
        ORDER BY cloud_provider, service_category
    """))
except Exception as e:
    print(f'Table not found or error: {e}')

## 5. Compare Silver (Athena) vs Gold (RDS)

In [ ]:
from local_helpers import athena_query

# Silver count from Athena
try:
    silver_counts = athena_query("""
        SELECT 'silver' as layer, COUNT(*) as cnt
        FROM silver_aws_custom_recommendations
    """)
    print('Silver (Athena):')
    display(silver_counts)
except Exception as e:
    print(f'Silver query failed: {e}')

# Gold count from RDS
gold_counts = rds_query("""
    SELECT 'gold' as layer, cloud_provider, COUNT(*) as cnt
    FROM custom_recommendations
    GROUP BY cloud_provider
""")
print('\nGold (RDS):')
display(gold_counts)

## 6. Ad-hoc RDS Queries

In [ ]:
# Your ad-hoc query here
rds_query("""
    SELECT 1 as placeholder
""")